# Symbolic verification of the remainder formulas in the non-degenerate case

This notebook verifies the explicit formulas for $\mathcal{R}_{a,b,\epsilon}^{\iota}$ given in Appendix A.1. Starting from the Rayleigh equation
$$
\left(\partial_y^2-k^2-\frac{b''(y)}{b(y)-\lambda+i\iota\epsilon}\right)\psi
=-\frac{\omega(y)}{b(y)-\lambda+i\iota\epsilon},\tag{1}
$$
we check that the singular terms $S_{a,b,\epsilon}^{\iota}$ and the remainder formulas $\mathcal{R}_{a,b,\epsilon}^{\iota}$ satisfy
$$
\left(\partial_y^2-k^2-\frac{b''(y)}{b(y)-\lambda+i\iota\epsilon}\right)
\left(\partial_\lambda^aD_\lambda^b\psi-S_{a,b,\epsilon}^{\iota}\right)
=\mathcal{R}_{a,b,\epsilon}^{\iota}.\tag{2}
$$

The singular terms $S_{a,b,\epsilon}^{\iota}$ and appendix remainders $\mathcal{R}_{a,b,\epsilon,\mathrm{appendix}}^{\iota}\,$ are transcribed directly from the Appendix manuscript. For each pair $(a,b)$, the notebook computes the left-hand side of (2) as $\mathcal{R}_{a,b,\epsilon,\mathrm{direct}}^{\iota}$ and compares the result with the entered appendix formula. Verification succeeds when the reduced difference is exactly zero:

$$
\mathcal{R}_{a,b,\epsilon,\mathrm{direct}}^{\iota}
-\mathcal{R}_{a,b,\epsilon,\mathrm{appendix}}^{\iota}\equiv0.
$$

## Verification procedure

*  **Step 1: Set up the notations and identities.**
    *   Define the functions, derivatives, Rayleigh operator, and reduction identities.
    *   All identities in this step follow from the Rayleigh equation and have been verified manually.

*  **Step 2: Enter the singular terms $S_{a,b,\epsilon}^\iota$.**
    *   The variable `Sab` represents $S_{a,b,\epsilon}^{\iota}$.
    *   When an alternative expression is used in the proof, it is entered as `Sab_rewrite`, and its difference from `Sab` is checked separately.

*  **Step 3: Compute the direct remainder $\mathcal{R}_{a,b,\epsilon,\text{direct}}^\iota$.**
    *   The expression for $\mathcal{R}_{a,b,\epsilon,\text{direct}}^\iota$ is computed using SymPy and stored as `Rab_direct`.
    *   The second- and higher-order spatial derivatives of $\psi$ are eliminated using the Rayleigh identities. The final expression is stored as `Rab_direct_reduced`.

*  **Step 4: Enter the appendix remainder $\mathcal{R}_{a,b,\epsilon,\text{appendix}}^\iota$.**
    *   The variable `Rab_appendix` represents $\mathcal{R}_{a,b,\epsilon,\text{appendix}}^\iota$ transcribed from Appendix A.1.
    *   The same reductions are applied to obtain the final expression `Rab_appendix_reduced`.

*  **Step 5: Compare the two expressions.**
    *   Two methods are used to compare the results.
        *   The first method uses SymPy's `.equals()` method. It may return `True`, `False`, or `None` (undetermined). A return value of `True` indicates that $\mathcal{R}_{a,b,\epsilon,\text{direct}}^\iota\equiv\mathcal{R}_{a,b,\epsilon,\text{appendix}}^\iota$.
        *   The second method simplifies their difference using SymPy's `sp.cancel` function. It returns zero if $\mathcal{R}_{a,b,\epsilon,\text{direct}}^\iota-\mathcal{R}_{a,b,\epsilon,\text{appendix}}^\iota\equiv 0$.

Throughout the procedure, all symbolic coefficients are kept exact.

## Notebook structure

The notebook is organized in the following order. We first set up all necessary variables, parameters, functions and operators. We then apply the procedure above to verify $\mathcal{R}_{a,b,\epsilon}^\iota$ for $(a,b)\in\{(1,0),(2,0), (1,1), (3,0), (2,1), (1,2)\}$ in order. The table below summarizes the purpose of each group of cells.

| Cells | 1-3 | 4-8 | 9-13 | 14-19 | 20-24 | 25-30 | 31-36 | 37 |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
|Content|Setup| Verify $\mathcal{R}_{1,0,\epsilon}^\iota$ | Verify $\mathcal{R}_{2,0,\epsilon}^\iota$ | Verify $\mathcal{R}_{1,1,\epsilon}^\iota$ | Verify $\mathcal{R}_{3,0,\epsilon}^\iota$ | Verify $\mathcal{R}_{2,1,\epsilon}^\iota$  | Verify $\mathcal{R}_{1,2,\epsilon}^\iota$ | Summary |

**Please run the notebook from top to bottom in a fresh kernel**, as later cases reuse singular terms and coefficient formulas defined in earlier cases. Each comparison prints a labeled result, and the last cell summarizes all checks.


In [1]:
# import packages

import sympy as sp

sp.init_printing()


# introduce functions and notations
y, lam = sp.symbols("y lambda", real=True)
k, epsilon, iota = sp.symbols("k epsilon iota")

b = sp.Function("b")(y)
psi = sp.Function("psi")(y, lam)
omega = sp.Function("omega")(y)
chi = sp.Function("chi")(y)


b1 = sp.diff(b, y)
b2 = sp.diff(b, y, 2)
b3 = sp.diff(b, y, 3)
b4 = sp.diff(b, y, 4)

psi_y = sp.diff(psi, y)
psi_lam = sp.diff(psi, lam)
psi_lam2 = sp.diff(psi, lam, 2)
Dpsi = psi_y + b1 * psi_lam
Dpsi_y = sp.diff(Dpsi, y)
D2psi = sp.diff(psi, y, 2) + b2 * psi_lam + sp.Integer(2) * b1 * sp.diff(psi_y, lam) + b1**2 * psi_lam2
D2psi_y = sp.diff(D2psi, y)
omega_y = sp.diff(omega, y)
omega_yy = sp.diff(omega, y, 2)
omega_yyy = sp.diff(omega, y, 3)
omega_yyyy = sp.diff(omega, y, 4)
chi_y = sp.diff(chi, y)
chi_yy = sp.diff(chi, y, 2)
denominator = b - lam + sp.I * iota * epsilon
log_denominator = sp.log(denominator)
rayleigh_numerator = b2 * psi - omega

In [2]:
# define Rayleigh operator
def rayleigh_operator(expression):
    return (
        sp.diff(expression, y, 2)
        - k**2 * expression
        - b2 * expression / denominator
    )

# Rayleigh equation for \partial_\lambda \psi
L_partial_lambda_psi = (
    (b2 * psi - omega) / denominator**2
)

# Rayleigh equation for \partial_\lambda^2 \psi
L_partial_lambda_2_psi = (
    2 * rayleigh_numerator / denominator**3
    + 2 * b2 * psi_lam / denominator**2
)

# Rayleigh equation for \partial_\lambda D_\lambda \psi
L_partial_lambda_Dpsi = (
    (b2 * Dpsi + b3 * psi - omega_y) / denominator**2
    + b3 * psi_lam / denominator
    + 2 * b2 * sp.diff(psi, y, lam, 2)
    + b3 * psi_lam2
)

# Rayleigh equation for \partial_lambda^3 \psi
L_partial_lambda_3_psi = (
    6 * rayleigh_numerator / denominator**4
    + 6 * b2 * psi_lam / denominator**3
    + 3 * b2 * psi_lam2 / denominator**2
)

# Rayleigh equation for \partial_\lambda^2 D_\lambda \psi
L_partial_lambda_2_Dpsi = (
    2 * (b2 * Dpsi + b3 * psi - omega_y) / denominator**3
    + 2 * (b2 * sp.diff(Dpsi, lam) + b3 * psi_lam) / denominator**2
    + b3 * psi_lam2 / denominator
    + 2 * b2 * sp.diff(psi, y, lam, 3)
    + b3 * sp.diff(psi, lam, 3)
)

# Rayleigh equation for \partial_\lambda D_\lambda^2 \psi
L_partial_lambda_D2psi = (
    (b2 * D2psi + 2 * b3 * Dpsi + b4 * psi - omega_yy) / denominator**2
    + (2 * b3 * sp.diff(Dpsi, lam) + b4 * psi_lam) / denominator
    + 4 * b2 * sp.diff(Dpsi, y, lam, 2)
    + 2 * b3 * sp.diff(Dpsi, lam, 2)
    - 2 * b2**2 * sp.diff(psi, lam, 3)
    + 2 * b3 * sp.diff(psi, y, lam, 2)
    + b4 * psi_lam2
)


In [3]:
# Rayleigh reductions

psi_yy_reduction = k**2 * psi + rayleigh_numerator / denominator

psi_yyy_reduction = (
    k**2 * psi_y
    + (b3 * psi + b2 * psi_y - omega_y) / denominator
    - b1 * rayleigh_numerator / denominator**2
)

psi_yyyy_reduction = sp.expand(
    sp.diff(psi_yyy_reduction, y).subs(sp.diff(psi, y, 2), psi_yy_reduction)
)

psi_lamyy_reduction = (
    k**2 * psi_lam
    + b2 * psi_lam / denominator
    + rayleigh_numerator / denominator**2
)

psi_lam2yy_reduction = (
    k**2 * psi_lam2
    + b2 * psi_lam2 / denominator
    + 2 * b2 * psi_lam / denominator**2
    + 2 * rayleigh_numerator / denominator**3
)

psi_lamyyy_reduction = sp.expand(sp.diff(psi_lamyy_reduction, y))

psi_lam3yy_reduction = (
    k**2 * sp.diff(psi, lam, 3)
    + b2 * sp.diff(psi, lam, 3) / denominator
    + 3 * b2 * psi_lam2 / denominator**2
    + 6 * b2 * psi_lam / denominator**3
    + 6 * rayleigh_numerator / denominator**4
)

psi_lam2yyy_reduction = sp.expand(sp.diff(psi_lam2yy_reduction, y))

psi_lamyyyy_reduction = sp.expand(sp.diff(psi_yyyy_reduction, lam))

def apply_raw_rayleigh_reductions(expression):
    replacements = (
        (sp.diff(psi, lam, y, 4), psi_lamyyyy_reduction),
        (sp.diff(psi, y, 4), psi_yyyy_reduction),
        (sp.diff(psi, lam, 2, y, 3), psi_lam2yyy_reduction),
        (sp.diff(psi, lam, y, 3), psi_lamyyy_reduction),
        (sp.diff(psi, lam, 3, y, 2), psi_lam3yy_reduction),
        (sp.diff(psi, lam, 2, y, 2), psi_lam2yy_reduction),
        (sp.diff(psi, lam, y, 2), psi_lamyy_reduction),
        (sp.diff(psi, y, 3), psi_yyy_reduction),
        (sp.diff(psi, y, 2), psi_yy_reduction),
    )
    for old, new in replacements:
        expression = expression.subs(old, new)
    return sp.expand(expression)

## Verification of $\mathcal{R}_{1,0,\epsilon}^\iota$


In [4]:
# Set coefficients and construct S_{1,0}
S10 = -(1-chi) * rayleigh_numerator / b1**2 * log_denominator

In [5]:
# Calculate R_{1,0}(y,lambda) directly.
R10_direct = sp.expand(L_partial_lambda_psi - rayleigh_operator(S10))

# apply reductions on R10_direct
R10_direct_reduced = apply_raw_rayleigh_reductions(R10_direct)

# display(sp.Eq(sp.Symbol("R10"), R10_direct))

In [6]:
# Hardcode R_{1,0}(y,lambda) from Appendix
c1_10 = sp.Integer(0)

c2_10 = (
    (
        2 * b1 * sp.diff((1-chi) / b1**2, y)
        + (1-chi) * b2 / b1**2
        ) * rayleigh_numerator
    + 2 * (1-chi) / b1 * (b3 * psi - omega_y)
)

mathfrak_c_10 = 2 * (1-chi) * b2 / b1

c3_10 = (
    sp.diff((1-chi) * b2 / b1**2, y, 2) * psi
    + 2 * sp.diff((1-chi) * b2 / b1**2, y) * psi_y
    - sp.diff((1-chi) / b1**2, y, 2) * omega
    - 2 * sp.diff((1-chi) / b1**2, y) * omega_y
    + (1-chi) / b1**2 * (k**2 * omega - omega_yy)
)

c4_10 = chi * rayleigh_numerator / denominator**2

R10_appendix = sp.expand(
    c1_10 * log_denominator / denominator
    + (c2_10 + mathfrak_c_10 * psi_y) / denominator
    + c3_10 * log_denominator
    + c4_10
)

# Apply reductions on R10_appendix.
R10_appendix_reduced = apply_raw_rayleigh_reductions(R10_appendix)

In [7]:
# Comparison 1: ask SymPy whether the reduced expressions are equal.
R10_matches_appendix = R10_direct_reduced.equals(R10_appendix_reduced)

print('For (a,b) = (1,0), SymPy .equals() returns:', R10_matches_appendix)

For (a,b) = (1,0), SymPy .equals() returns: True


In [8]:
# Comparison 2: compute the exact residual.
R10_difference = sp.cancel(R10_direct_reduced - R10_appendix_reduced)

print('For (a,b) = (1,0), the reduced direct-minus-appendix residual is:', R10_difference)

For (a,b) = (1,0), the reduced direct-minus-appendix residual is: 0


## Verification of $\mathcal{R}_{2,0,\epsilon}^\iota$

In [9]:
# Set coefficients for S_{2,0}.
C1_20 = 1 / b1**2

C2_20 = -2 * b2 / b1**3

C3_20 = -2 * b3 / b1**3 + 3 * b2**2 / b1**4

C4_20 = 2 / b1**3

C5_20 = -3 * b2 / b1**4

# S_{2,0} construction
S20 = (
    C1_20 * (1-chi) * rayleigh_numerator / denominator
    + (
        C2_20 * Dpsi
        + C3_20 * psi
        + C4_20 * omega_y
        + C5_20 * omega
        ) * (1-chi) * log_denominator
)

In [10]:
# Calculate R_{2,0}(y,lambda) directly.
R20_direct = sp.expand(L_partial_lambda_2_psi - rayleigh_operator(S20))

# apply reductions on R20_direct
R20_direct_reduced = apply_raw_rayleigh_reductions(R20_direct)

# display(sp.Eq(sp.Symbol("R20"), R20_direct))

In [11]:
# Hardcode R_{2,0}(y, lambda) from the appendix.
c1_20 = (
    (1-chi) * 2 * (b1 * b3 - 2 * b2**2) / b1**4
    * rayleigh_numerator
)

c2_20 = (
    (
        -sp.diff((1-chi) * b2 / b1**2, y, 2)
        - 2 * b1 * sp.diff((1-chi) * (
            - 2 * b3 / b1**3
            + 3 * b2**2 / b1**4
            ), y)
        - (1-chi) * b2 * (
            - 2 * b3 / b1**3
            + 3 * b2**2 / b1**4
            )
        ) * psi
    + (
        4 * b1 * sp.diff((1-chi) * b2 / b1**3, y)
        + 2 * (1-chi) * b2**2 / b1**3
        ) * Dpsi
    + sp.diff((1-chi) * omega / b1**2, y, 2)
    - k**2 * (1-chi) * omega / b1**2
    - 2 * b1 * sp.diff((1-chi) * (
        2 * omega_y / b1**3
        - 3 * b2 * omega / b1**4
        ), y)
    - (1-chi) * b2 * (
        2 * omega_y / b1**3
        - 3 * b2 * omega / b1**4
        )
)

mathfrak_c1_20 = 4 * (1-chi) * b2 / b1**2

mathfrak_c2_20 = (
    -2 * sp.diff((1-chi) * b2 / b1**2, y)
    + (1-chi) * (
        4 * b3 / b1**2
        - 6 * b2**2 / b1**3
        )
)

c3_20 = (
    -sp.diff((1-chi) * (
        2 * omega_y / b1**3
        - 3 * b2 * omega / b1**4
        ), y, 2)
    + k**2 * (1-chi) * (
        2 * omega_y / b1**3
        - 3 * b2 * omega / b1**4
        )
    -(
        sp.diff((1-chi) * (
            - 2 * b3 / b1**3
            + 3 * b2**2 / b1**4
            ), y, 2)
        + 4 * k**2 * (1-chi) * b2**2 / b1**4
        ) * psi
    -(
        2 * sp.diff((1-chi) * (
            - 2 * b3 / b1**3
            + 3 * b2**2 / b1**4
            ), y)
        + 2 * (1-chi) * b2 / b1**3 * (
            b3 / b1 - 2 * b2**2 / b1**2
            )
        ) * psi_y
    + (
        sp.diff(2 * (1-chi) * b2 / b1**3, y, 2)
        + 2 * (1-chi) * b2 / b1**3 * (
            b3 / b1 - 2 * b2**2 / b1**2
            )
        ) * Dpsi
    + (
        4 * sp.diff((1-chi) * b2 / b1**3, y)
        + 4 * (1-chi) * b2**2 / b1**4
        ) * Dpsi_y
)

c4_20 = (
    chi * (
        2 * rayleigh_numerator / denominator**3
        + 2 * b2 * psi_lam / denominator**2
        )
    - chi_y * 2 * rayleigh_numerator / (b1 * denominator**2)
)

R20_appendix = sp.expand(
    c1_20 * log_denominator / denominator
    + (
        c2_20
        + mathfrak_c1_20 * Dpsi_y
        + mathfrak_c2_20 * psi_y
        ) / denominator
    + c3_20 * log_denominator
    + c4_20
)

# Apply reductions on R20_appendix.
R20_appendix_reduced = apply_raw_rayleigh_reductions(R20_appendix)

In [12]:
# Comparison 1: ask SymPy whether the reduced expressions are equal.
R20_matches_appendix = R20_direct_reduced.equals(R20_appendix_reduced)

print('For (a,b) = (2,0), SymPy .equals() returns:', R20_matches_appendix)

For (a,b) = (2,0), SymPy .equals() returns: True


In [13]:
# Comparison 2: compute the exact residual.
R20_difference = sp.cancel(R20_direct_reduced - R20_appendix_reduced)

print('For (a,b) = (2,0), the reduced direct-minus-appendix residual is:', R20_difference)

For (a,b) = (2,0), the reduced direct-minus-appendix residual is: 0


## Verification of $\mathcal{R}_{1,1,\epsilon}^\iota$

In [14]:
# Set coefficients for S_{1,1}.
C1_11 = -b2 / b1**2

C2_11 = -b3 / b1**2 + 2 * b2**2 / b1**3

C3_11 = 1 / b1**2

C4_11 = -2 * b2 / b1**3

# Construct S_{1,1}
S11 = (
    C1_11 * Dpsi
    + C2_11 * psi
    + C3_11 * omega_y
    + C4_11 * omega
) * (1-chi) * log_denominator

In [15]:
# Construct S_{1,1}_rewrite.

S11_rewrite = (
    b1 * S20
    +(-b2 / b1 * psi + omega / b1) * (1 - chi) / denominator
    + (
        -omega_y / b1**2
        +(b3 / b1**2 - b2**2 / b1**3) * psi
        +b2 / b1**3 * omega
        +b2 / b1**2 * Dpsi
    ) * (1-chi) * log_denominator
)

S11_rewrite_difference = sp.cancel(S11_rewrite - S11)
print('For (a,b) = (1,1), the rewritten-minus-original singular-term residual is:', S11_rewrite_difference)

For (a,b) = (1,1), the rewritten-minus-original singular-term residual is: 0


In [16]:
# Calculate R_{1,1}(y,lambda) directly.
R11_direct = sp.expand(L_partial_lambda_Dpsi - rayleigh_operator(S11))

# apply reductions on R11_direct
R11_direct_reduced = apply_raw_rayleigh_reductions(R11_direct)

# display(sp.Eq(sp.Symbol("R11"), R11_direct))

In [17]:
# Hardcode R_{1,1}(y, lambda) from the appendix.
c1_11 = (1-chi) * (
    (b1 * b3 - 2 * b2**2) / b1**3
) * rayleigh_numerator

c2_11 = (
    (
        sp.diff((1-chi) * b2 / b1, y, 2)
        - 2 * b1 * sp.diff((1-chi) * (
            b3 / b1**2
            - b2**2 / b1**3
            ), y)
        - (1-chi) * b2 * (
            b3 / b1**2
            - b2**2 / b1**3
            )
        ) * psi
    + (
        (1-chi) * b3 / b1
        - 2 * b1 * sp.diff((1-chi) * b2 / b1**2, y)
        - (1-chi) * b2**2 / b1**2
        ) * Dpsi
    - sp.diff((1-chi) * omega / b1, y, 2)
    + k**2 * (1-chi) * omega / b1
    + 2 * b1 * sp.diff((1-chi) * (
        omega_y / b1**2
        - b2 * omega / b1**3
        ), y)
    + (1-chi) * b2 * (
        omega_y / b1**2
        - b2 * omega / b1**3
        )
    + b1 * c2_20
)

mathfrak_c1_11 = 2 * (1-chi) * b2 / b1

mathfrak_c2_11 = (1-chi) * (
    b3 / b1
    - 2 * b2**2 / b1**2
)

c3_11 = (
    b1 * c3_20
    + sp.diff((1-chi) * (
        omega_y / b1**2
        - b2 * omega / b1**3
        ), y, 2)
    - k**2 * (1-chi) * (
        omega_y / b1**2
        - b2 * omega / b1**3
        )
    -(
        sp.diff((1-chi) * (
            b3 / b1**2
            - b2**2 / b1**3
            ), y, 2)
        - 2 * k**2 * (1-chi) * b2**2 / b1**3
        ) * psi
    + (
        -2 * sp.diff((1-chi) * (
            b3 / b1**2
            - b2**2 / b1**3
            ), y)
        + (1-chi) * b2 / b1**2 * (
            b3 / b1
            - 2 * b2**2 / b1**2
            )
        ) * psi_y
    -(
        sp.diff((1-chi) * b2 / b1**2, y, 2)
        + (1-chi) * b2 / b1**2 * (
            b3 / b1
            - 2 * b2**2 / b1**2
            )
        ) * Dpsi
    -(
        2 * sp.diff((1-chi) * b2 / b1**2, y)
        + 2 * (1-chi) * b2**2 / b1**3
        ) * Dpsi_y
)

c4_11 = (
    chi * (
        (b2 * Dpsi + b3 * psi - omega_y) / denominator**2
        + b3 * psi_lam / denominator
        )
    + 2 * b2 * sp.diff(psi_lam2 - S20, y)
    + b3 * (psi_lam2 - S20)
)

R11_appendix = sp.expand(
    c1_11 * log_denominator / denominator
    + (
        c2_11
        + mathfrak_c1_11 * Dpsi_y
        + mathfrak_c2_11 * psi_y
        ) / denominator
    + c3_11 * log_denominator
    + c4_11
)

# Apply reductions on R11_appendix.
R11_appendix_reduced = apply_raw_rayleigh_reductions(R11_appendix)

In [18]:
# Comparison 1: ask SymPy whether the reduced expressions are equal.
R11_matches_appendix = R11_direct_reduced.equals(R11_appendix_reduced)

print('For (a,b) = (1,1), SymPy .equals() returns:', R11_matches_appendix)

For (a,b) = (1,1), SymPy .equals() returns: True


In [19]:
# Comparison 2: compute the exact residual.
R11_difference = sp.cancel(R11_direct_reduced - R11_appendix_reduced)

print('For (a,b) = (1,1), the reduced direct-minus-appendix residual is:', R11_difference)

For (a,b) = (1,1), the reduced direct-minus-appendix residual is: 0


## Verification of $\mathcal{R}_{3,0,\epsilon}^\iota$

In [20]:
# set coefficients for S_{3,0}
C1 = 1 / b1**2
C2 = -b2 / b1**3
C3 = 3 * b2 / b1**3
C4 = 2 * b3 / b1**3 - 3 * b2**2 / b1**4
C5 = -2 / b1**3
C6 = 3 * b2 / b1**4
C7 = -3 * b2 / b1**4
C8 = -6 * b3 / b1**4 + 12 * b2**2 / b1**5
C9 = -3 * b4 / b1**4 + 16 * b2 * b3 / b1**5 - 15 * b2**3 / b1**6 - k**2 * b2 / b1**4
C10 = 3 / b1**4
C11 = -12 * b2 / b1**5
C12 = k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6

# S_{3,0} construction
S30 = (
    C1 * (1 - chi) * rayleigh_numerator / denominator**2
    + (C2 * psi_y + C3 * Dpsi + C4 * psi + C5 * omega_y + C6 * omega)
      * (1 - chi) / denominator
    + (C7 * D2psi + C8 * Dpsi + C9 * psi
       + C10 * omega_yy + C11 * omega_y + C12 * omega)
      * (1 - chi) * log_denominator
)

In [21]:
# Calculate R_{3,0}(y,lambda) directly.
R30_direct = sp.expand(L_partial_lambda_3_psi - rayleigh_operator(S30))

# apply reductions on R30_direct
R30_direct_reduced = apply_raw_rayleigh_reductions(R30_direct)

# display(sp.Eq(sp.Symbol("R30"), R30_direct))

In [22]:
# Hardcode R_{3,0}(y,lambda) from documents
c1_30 = (1 - chi) * (
    (
        (3 * b2 * b4 + 6 * b3**2 ) / b1**4
        - 42 * b2**2 * b3 / b1**5
        + 42 * b2**4 / b1**6
        ) * psi
    + (
        - 3 * b4 / b1**4
        + 30 * b2 * b3 / b1**5
        - 42 * b2**3 / b1**6
        ) * omega
    + (
        - 6 * b3 / b1**4
        + 12 * b2**2 / b1**5
        ) * omega_y
    + (
        6 * b2 * b3 / b1**4
        - 12 * b2**3 / b1**5
        ) * Dpsi
)

c2_30 = (
    -(
        2 * k**2 * sp.diff(-(1-chi) * b2 / b1**3, y)
        + sp.diff((1-chi) * (2 * b3 / b1**3 - 3 * b2**2 / b1**4), y, 2)
        - k**2 * (2 * b2 / b1) * (3 * (1-chi) * b2 / b1**3)
        +2 * b1 * sp.diff((1-chi) * (
            -3 * b4 / b1**4
            + 16 * b2 * b3 / b1**5
            - 15 * b2**3 / b1**6
            - k**2 * b2 / b1**4), y)
        + b2 * ((1-chi) * (
            - 3 * b4 / b1**4
            + 16 * b2 * b3 / b1**5
            - 15 * b2**3 / b1**6
            - k**2 * b2 / b1**4
            ))
        ) * psi
    -(
        sp.diff(3 * (1-chi) * b2 / b1**3, y, 2)
        + (b3 / b1 - 2 * b2**2 / b1**2) * (3 * (1-chi) * b2 / b1**3)
        + 2 * b1 * sp.diff((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5), y)
        + b2 * ((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5))
        ) * Dpsi
    -(
        2 * b1 * sp.diff(-3 * (1-chi) * b2 / b1**4, y)
        + b2 * (-3 * (1-chi) * b2 / b1**4)
        ) * D2psi
    -(
        (-2 * (1-chi) / b1**3) + 2 * b1 * (3 * (1-chi) / b1**4)
        ) * omega_yyy
    -(
        2 * sp.diff(-2 * (1-chi) / b1**3, y)
        + 3 * (1-chi) * b2 / b1**4
        + 2 * b1 * (-12 * (1-chi) * b2 / b1**5 + sp.diff(3 * (1-chi) / b1**4, y))
        + b2 * (3 * (1-chi) / b1**4)
        ) * omega_yy
    -(
        sp.diff(-2 * (1-chi) / b1**3, y, 2)
        - k**2 * (-2 * (1-chi) / b1**3)
        + 2 * sp.diff(3 * (1-chi) * b2 / b1**4, y)
        + 2 * b1 * (
            sp.diff(-12 * (1-chi) * b2 / b1**5, y)
            + (1-chi) * (k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6)
            )
        + b2 * (-12 * (1-chi) * b2 / b1**5)
        ) * omega_y
    -(
        sp.diff(3 * (1-chi) * b2 / b1**4, y, 2)
        - k**2 * (3 * (1-chi) * b2 / b1**4)
        + 2 * b1 * sp.diff((1-chi) * (k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6), y)
        + b2 * ((1-chi) * (k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6))
        ) * omega
)

mathfrak_c1_30 = -2 * b1 * (-3 * (1-chi) * b2 / b1**4)

mathfrak_c2_30 = (
    -2 * sp.diff(3 * (1-chi) * b2 / b1**3, y)
    - (2 * b2 / b1) * (3 * (1-chi) * b2 / b1**3)
    - 2 * b1 * ((1-chi) * (
        - 6 * b3 / b1**4
        + 12 * b2**2 / b1**5
        ))
)

mathfrak_c3_30 = (
    -sp.diff(-(1-chi) * b2 / b1**3, y, 2)
    - 2 * sp.diff((1-chi) * (
        2 * b3 / b1**3
        - 3 * b2**2 / b1**4
        ), y)
    + (b3 / b1 - 2 * b2**2 / b1**2) * (3 * (1-chi) * b2 / b1**3)
    - 2 * b1 * ((1-chi) * (
        - 3 * b4 / b1**4
        + 16 * b2 * b3 / b1**5
        - 15 * b2**3 / b1**6
        - k**2 * b2 / b1**4
        ))
)

c3_30 = (
    -(
        sp.diff((1-chi) * (
            - 3 * b4 / b1**4
            + 16 * b2 * b3 / b1**5
            - 15 * b2**3 / b1**6
            - k**2 * b2 / b1**4
            ), y, 2)
        - k**2 * (2 * b2 / b1) * ((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5))
        - k**2 * (2 * b3 / b1 - 6 * b2**2 / b1**2) * (-3 * (1-chi) * b2 / b1**4)
        ) * psi
    -(
        2 * sp.diff((1-chi) * (
            - 3 * b4 / b1**4
            + 16 * b2 * b3 / b1**5
            - 15 * b2**3 / b1**6
            - k**2 * b2 / b1**4
            ), y)
        - (b3 / b1 - 2 * b2**2 / b1**2) * ((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5))
        - (b4 / b1 - 6 * b2 * b3 / b1**2 + 6 * b2**3 / b1**3) * (-3 * (1-chi) * b2 / b1**4)
        ) * psi_y
    -(
        sp.diff((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5), y, 2)
        + (b3 / b1 - 2 * b2**2 / b1**2) * ((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5))
        + (
            (b4 / b1 - 6 * b2 * b3 / b1**2 + 6 * b2**3 / b1**3)
            - 2 * k**2 * (2 * b2 / b1)
            ) * (-3 * (1-chi) * b2 / b1**4)
        ) * Dpsi
    -(
        2 * sp.diff((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5), y)
        + (2 * b2 / b1) * ((1-chi) * (- 6 * b3 / b1**4 + 12 * b2**2 / b1**5))
        ) * Dpsi_y
    -(
        sp.diff(-3 * (1-chi) * b2 / b1**4, y, 2)
        + (2 * b3 / b1 - 6 * b2**2 / b1**2) * (-3 * (1-chi) * b2 / b1**4)
        ) * D2psi
    -(
        2 * sp.diff(-3 * (1-chi) * b2 / b1**4, y)
        + 2 * (2 * b2 / b1) * (-3 * (1-chi) * b2 / b1**4)
        ) * D2psi_y
    -(
        3 * (1-chi) / b1**4
        ) * omega_yyyy
    -(
        -12 * (1-chi) * b2 / b1**5
        + 2 * sp.diff(3 * (1-chi) / b1**4, y)
        ) * omega_yyy
    -(
        2 * sp.diff(-12 * (1-chi) * b2 / b1**5, y)
        + (1-chi) * (k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6)
        + sp.diff(3 * (1-chi) / b1**4, y, 2)
        - k**2 * (3 * (1-chi) / b1**4)
        ) * omega_yy
    -(
        sp.diff(-12 * (1-chi) * b2 / b1**5, y, 2)
        - k**2 * (-12 * (1-chi) * b2 / b1**5)
        + 2 * sp.diff((1-chi) * (k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6), y)
        ) * omega_y
    -(
        sp.diff((1-chi) * (k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6), y, 2)
        - k**2 * ((1-chi) * (k**2 / b1**4 - 4 * b3 / b1**5 + 15 * b2**2 / b1**6))
        ) * omega
)

c4_30 = (
    chi * (
        6 * rayleigh_numerator / denominator**4
        + 6 * b2 * psi_lam / denominator**3
        + 3 * b2 * psi_lam2 / denominator**2
    )
    + chi_y * (
        -4 * rayleigh_numerator / (b1 * denominator**3)
        + 2 / denominator**2 * (
            -b3 / b1**2 * psi
            + 2 * b2 / b1**2 * psi_y
            + omega_y / b1**2
            - 3 * b2 / b1**2 * Dpsi
        )
    )
    + chi_yy * rayleigh_numerator / (b1**2 * denominator**2)
)

R30_appendix = sp.expand(
    c1_30 * log_denominator/denominator
    + (
        c2_30
        + mathfrak_c1_30 * D2psi_y
        + mathfrak_c2_30 * Dpsi_y
        + mathfrak_c3_30 * psi_y
      ) / denominator
    + c3_30 * log_denominator
    + c4_30
)

# apply reductions on R30_appendix
R30_appendix_reduced = apply_raw_rayleigh_reductions(R30_appendix)

In [23]:
# Comparison 1: ask SymPy whether the reduced expressions are equal.
R30_matches_appendix = R30_direct_reduced.equals(R30_appendix_reduced)

print('For (a,b) = (3,0), SymPy .equals() returns:', R30_matches_appendix)

For (a,b) = (3,0), SymPy .equals() returns: True


In [24]:
# Comparison 2: compute the exact residual.
R30_difference = sp.cancel(R30_direct_reduced - R30_appendix_reduced)

print('For (a,b) = (3,0), the reduced direct-minus-appendix residual is:', R30_difference)

For (a,b) = (3,0), the reduced direct-minus-appendix residual is: 0


## Verification of $\mathcal{R}_{2,1,\epsilon}^\iota$

In [25]:
# Set coefficients and construct S_{2,1}.
C1_21 = b2 / b1**2

C2_21 = b3 / b1**2 - 2 * b2**2 / b1**3

C3_21 = -1 / b1**2

C4_21 = 2 * b2 / b1**3

C5_21 = -2 * b2 / b1**3

C6_21 = -4 * b3 / b1**3 + 9 * b2**2 / b1**4

C7_21 = (
    -2 * b4 / b1**3
    + 12 * b2 * b3 / b1**4
    - 12 * b2**3 / b1**5
)

C8_21 = 2 / b1**3

C9_21 = -9 * b2 / b1**4

C10_21 = -3 * b3 / b1**4 + 12 * b2**2 / b1**5

# Construct S_{2,1}.
S21 = (
    (C1_21 * Dpsi + C2_21 * psi + C3_21 * omega_y + C4_21 * omega)
    * (1-chi) / denominator
    + (
        C5_21 * D2psi
        + C6_21 * Dpsi
        + C7_21 * psi
        + C8_21 * omega_yy
        + C9_21 * omega_y
        + C10_21 * omega
    ) * (1-chi) * log_denominator
)

In [26]:
# Construct S_{2,1}_rewrite.

S21_rewrite = (
    b1 * S30
    +(-b2 / b1 * psi + omega / b1) * (1 - chi) / denominator**2
    +(
        b2 / b1**2 * psi_y + omega_y / b1**2
        + (-b3 / b1**2 + b2**2 / b1**3) * psi
        - b2 / b1**3 * omega - 2 * b2 / b1**2 * Dpsi
    ) * (1-chi) / denominator
    + (
        3 * b2 * omega_y / b1**4
        + (k**2 * b1**2 * b2 + b1**2 * b4 - 4 * b1 * b2 * b3 + 3 * b2**3) * psi / b1**5
        - (k**2 * b1**2 - b1 * b3 + 3 * b2**2) * omega / b1**5
        + (2 * b1 * b3 - 3 * b2**2) * Dpsi / b1**4
        + b2 * D2psi / b1**3 - omega_yy / b1**3
    ) * (1-chi) * log_denominator
)

S21_rewrite_difference = sp.cancel(S21_rewrite - S21)
print('For (a,b) = (2,1), the rewritten-minus-original singular-term residual is:', S21_rewrite_difference)

For (a,b) = (2,1), the rewritten-minus-original singular-term residual is: 0


In [27]:
# Calculate R_{2,1}(y, lambda) directly.
R21_direct = sp.expand(L_partial_lambda_2_Dpsi - rayleigh_operator(S21))

# Apply the Rayleigh reductions.
R21_direct_reduced = apply_raw_rayleigh_reductions(R21_direct)

In [28]:
# Hardcode R_{2,1}(y, lambda) from the appendix.
c1_21 = (
    b1 * c1_30
    - (
        (1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4
        * (b3 - 2 * b2**2 / b1)
        + (1 - chi) * b2 / b1**3
        * (b4 - 6 * b2 * b3 / b1 + 6 * b2**3 / b1**2)
    ) * psi
    - (
        (1 - chi) * b2 / b1**3 * (2 * b3 - 4 * b2**2 / b1)
    ) * Dpsi
    + (
        b2 * (3 * (1 - chi) * b2 / b1**4)
        + (1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4
        - (4 * b2 / b1) * ((1 - chi) * b2 / b1**3)
    ) * omega_y
    + (
        (1 - chi) * (
            k**2 * b1**2 * b2
            + b1**2 * b4
            - 4 * b1 * b2 * b3
            + 3 * b2**3
        ) / b1**5
        + b2 * (
            -(1 - chi) * (k**2 * b1**2 - b1 * b3 + 3 * b2**2) / b1**5
        )
        - (2 * b2 / b1) * (1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4
        - (2 * b3 / b1 - 6 * b2**2 / b1**2) * ((1 - chi) * b2 / b1**3)
    ) * omega
)

c2_21 = (
    b1 * c2_30
    + (
        (1 - chi) * b3 / b1**2
        - 2 * b1 * sp.diff((1 - chi) * b2 / b1**3, y)
        - b2 * ((1 - chi) * b2 / b1**3)
    ) * D2psi
    + (
        (1 - chi) * b2 * b3 / b1**3
        + 2 * ((1 - chi) * b2 / b1**2) * (b3 / b1 - 2 * b2**2 / b1**2)
        + 2 * sp.diff((1 - chi) * b2 / b1**2, y, 2)
        - 2 * b1 * sp.diff((1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4, y)
        - b2 * (1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4
    ) * Dpsi
    + (
        k**2 * (1 - chi) * b3 / b1**2
        - 2 * k**2 * sp.diff((1 - chi) * b2 / b1**2, y)
        - sp.diff((1 - chi) * (-b3 / b1**2 + b2**2 / b1**3), y, 2)
        - (4 * k**2 * b2 / b1) * ((1 - chi) * b2 / b1**2)
        - 2 * b1 * sp.diff((1 - chi) * (
            k**2 * b1**2 * b2
            + b1**2 * b4
            - 4 * b1 * b2 * b3
            + 3 * b2**3
        ) / b1**5, y)
        - b2 * (1 - chi) * (
            k**2 * b1**2 * b2
            + b1**2 * b4
            - 4 * b1 * b2 * b3
            + 3 * b2**3
        ) / b1**5
    ) * psi
    + (
        -(1 - chi) / b1**2
        + 2 * b1 * (1 - chi) / b1**3
    ) * omega_yyy
    + (
        -2 * sp.diff((1 - chi) / b1**2, y)
        + (1 - chi) * b2 / b1**3
        + 2 * b1 * sp.diff((1 - chi) / b1**3, y)
        - 2 * b1 * (3 * (1 - chi) * b2 / b1**4)
        + b2 * ((1 - chi) / b1**3)
    ) * omega_yy
    + (
        k**2 * (1 - chi) / b1**2
        - sp.diff((1 - chi) / b1**2, y, 2)
        + 2 * sp.diff((1 - chi) * b2 / b1**3, y)
        - 2 * b1 * sp.diff(3 * (1 - chi) * b2 / b1**4, y)
        - 2 * b1 * (
            -(1 - chi) * (k**2 * b1**2 - b1 * b3 + 3 * b2**2) / b1**5
        )
        - b2 * (3 * (1 - chi) * b2 / b1**4)
    ) * omega_y
    + (
        -k**2 * (1 - chi) * b2 / b1**3
        + sp.diff((1 - chi) * b2 / b1**3, y, 2)
        - 2 * b1 * sp.diff(
            -(1 - chi) * (k**2 * b1**2 - b1 * b3 + 3 * b2**2) / b1**5, y
        )
        - b2 * (
            -(1 - chi) * (k**2 * b1**2 - b1 * b3 + 3 * b2**2) / b1**5
        )
    ) * omega
)

mathfrak_c1_21 = (
    b1 * mathfrak_c1_30
    - 2 * b1 * ((1 - chi) * b2 / b1**3)
)

mathfrak_c2_21 = (
    b1 * mathfrak_c2_30
    - 2 * (1 - chi) * b3 / b1**2
    + (4 * b2 / b1) * ((1 - chi) * b2 / b1**2)
    + 4 * sp.diff((1 - chi) * b2 / b1**2, y)
    - 2 * b1 * (1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4
)

mathfrak_c3_21 = (
    b1 * mathfrak_c3_30
    - (1 - chi) * b2 * b3 / b1**3
    - sp.diff((1 - chi) * b2 / b1**2, y, 2)
    - 2 * sp.diff((1 - chi) * (-b3 / b1**2 + b2**2 / b1**3), y)
    - 2 * ((1 - chi) * b2 / b1**2) * (b3 / b1 - 2 * b2**2 / b1**2)
    - 2 * b1 * (1 - chi) * (
        k**2 * b1**2 * b2
        + b1**2 * b4
        - 4 * b1 * b2 * b3
        + 3 * b2**3
    ) / b1**5
)

c3_21 = (
    b1 * c3_30
    - (3 * (1 - chi) * b2 / b1**4) * (omega_yyy - k**2 * omega_y)
    - 2 * sp.diff(3 * (1 - chi) * b2 / b1**4, y) * omega_yy
    - sp.diff(3 * (1 - chi) * b2 / b1**4, y, 2) * omega_y
    - 2 * sp.diff((1 - chi) * (
        k**2 * b1**2 * b2
        + b1**2 * b4
        - 4 * b1 * b2 * b3
        + 3 * b2**3
    ) / b1**5, y) * psi_y
    - sp.diff((1 - chi) * (
        k**2 * b1**2 * b2
        + b1**2 * b4
        - 4 * b1 * b2 * b3
        + 3 * b2**3
    ) / b1**5, y, 2) * psi
    - (
        -(1 - chi) * (k**2 * b1**2 - b1 * b3 + 3 * b2**2) / b1**5
    ) * (omega_yy - k**2 * omega)
    - 2 * sp.diff(
        -(1 - chi) * (k**2 * b1**2 - b1 * b3 + 3 * b2**2) / b1**5, y
    ) * omega_y
    - sp.diff(
        -(1 - chi) * (k**2 * b1**2 - b1 * b3 + 3 * b2**2) / b1**5, y, 2
    ) * omega
    - ((1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4) * (
        (b3 / b1 - 2 * b2**2 / b1**2) * (Dpsi - psi_y)
        + (2 * b2 / b1) * (Dpsi_y - k**2 * psi)
    )
    - 2 * sp.diff((1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4, y) * Dpsi_y
    - sp.diff((1 - chi) * (2 * b1 * b3 - 3 * b2**2) / b1**4, y, 2) * Dpsi
    - (4 * b2 / b1) * ((1 - chi) * b2 / b1**3) * (
        D2psi_y - k**2 * Dpsi
        + (-b3 / b1 + 2 * b2**2 / b1**2) * (Dpsi - psi_y)
        - (2 * b2 / b1) * (Dpsi_y - k**2 * psi)
    )
    - ((1 - chi) * b2 / b1**3) * (
        (-4 * b2**2 / b1**2 + 2 * b3 / b1) * (D2psi - Dpsi_y)
        + (2 * b3 / b1) * (Dpsi_y - k**2 * psi)
    )
    + (2 * b2**2 / b1**2) * ((1 - chi) * b2 / b1**3) * (
        D2psi - 2 * Dpsi_y + k**2 * psi
        + (b2 / b1) * (Dpsi - psi_y)
    )
    - ((1 - chi) * b2 / b1**3) * (-2 * b2 * b3 / b1**2 + b4 / b1) * (Dpsi - psi_y)
    - 2 * sp.diff((1 - chi) * b2 / b1**3, y) * D2psi_y
    - sp.diff((1 - chi) * b2 / b1**3, y, 2) * D2psi
    + ((1 - chi) / b1**3) * (omega_yyyy - k**2 * omega_yy)
    + 2 * sp.diff((1 - chi) / b1**3, y) * omega_yyy
    + sp.diff((1 - chi) / b1**3, y, 2) * omega_yy
)

r30 = sp.diff(psi, lam, 3) - S30

c4_21 = (
    2 * b2 * sp.diff(r30, y)
    + b3 * r30
    + chi * (
        2 * (b2 * Dpsi + b3 * psi - omega_y) / denominator**3
        + 2 * (b2 * sp.diff(Dpsi, lam) + b3 * psi_lam) / denominator**2
        + b3 * psi_lam2 / denominator
    )
    + 2 * chi_y / denominator**2 * (
        -(b2 * Dpsi + b3 * psi - omega_y) / b1
        + b2 * rayleigh_numerator / b1**2
    )
)

R21_appendix = (
    c1_21 * log_denominator / denominator
    + (
        c2_21
        + mathfrak_c1_21 * D2psi_y
        + mathfrak_c2_21 * Dpsi_y
        + mathfrak_c3_21 * psi_y
    ) / denominator
    + c3_21 * log_denominator
    + c4_21
)

# Apply reductions on R21_appendix.
R21_appendix_reduced = apply_raw_rayleigh_reductions(R21_appendix)

In [29]:
# Comparison 1: ask SymPy whether the reduced expressions are equal.
R21_matches_appendix = R21_direct_reduced.equals(R21_appendix_reduced)

print('For (a,b) = (2,1), SymPy .equals() returns:', R21_matches_appendix)

For (a,b) = (2,1), SymPy .equals() returns: True


In [30]:
# Comparison 2: compute the exact residual.
R21_difference = sp.cancel(R21_direct_reduced - R21_appendix_reduced)

print('For (a,b) = (2,1), the reduced direct-minus-appendix residual is:', R21_difference)

For (a,b) = (2,1), the reduced direct-minus-appendix residual is: 0


## Verification of $\mathcal{R}_{1,2,\epsilon}^\iota$

In [31]:
# Set coefficients for S_{1,2}.
C1_12 = -b2 / b1**2

C2_12 = (
    -2 * b3 / b1**2
    + 4 * b2**2 / b1**3
)

C3_12 = (
    -b4 / b1**2
    + 6 * b2 * b3 / b1**3
    - 6 * b2**3 / b1**4
)

C4_12 = 1 / b1**2

C5_12 = -4 * b2 / b1**3

C6_12 = (
    -2 * b3 / b1**3
    + 6 * b2**2 / b1**4
)

# Construct S_{1,2}.
S12 = (
    C1_12 * D2psi
    + C2_12 * Dpsi
    + C3_12 * psi
    + C4_12 * omega_yy
    + C5_12 * omega_y
    + C6_12 * omega
) * (1 - chi) * log_denominator

In [32]:
# Construct S_{1,2}_rewrite.

S12_rewrite = (
    - b1**2 * S30 + rayleigh_numerator * (1-chi) / denominator**2
    + 2 * b1 * S21 + b2 * S20
    + b2 * (Dpsi - psi_y) * (1-chi) / (b1 * denominator)
    - k**2 * rayleigh_numerator / b1**2 * (1-chi) * log_denominator
)

S12_rewrite_difference = sp.cancel(S12_rewrite - S12)
print('For (a,b) = (1,2), the rewritten-minus-original singular-term residual is:', S12_rewrite_difference)

For (a,b) = (1,2), the rewritten-minus-original singular-term residual is: 0


In [33]:
# Calculate R_{1,2}(y, lambda) directly.
R12_direct = sp.expand(L_partial_lambda_D2psi - rayleigh_operator(S12))

# Apply the Rayleigh reductions.
R12_direct_reduced = apply_raw_rayleigh_reductions(R12_direct)

In [34]:
# Hardcode R_{1,2}(y, lambda) from the appendix.
r30 = sp.diff(psi, lam, 3) - S30

r21 = sp.diff(Dpsi, lam, 2) - S21

r20 = psi_lam2 - S20

c1_12 = (
    -b1**2 * c1_30
    + 2 * b1 * c1_21
    + b2 * c1_20
)

c2_12 = (
    -b1**2 * c2_30
    + 2 * b1 * c2_21
    + b2 * c2_20
    + 2 * k**2 * b1 * (
        sp.diff((1 - chi) / b1**2, y) * rayleigh_numerator
        + (1 - chi) / b1**2 * (b3 * psi - omega_y)
    )
    + k**2 * (1 - chi) / b1**2 * b2 * rayleigh_numerator
)

mathfrak_c1_12 = (
    -b1**2 * mathfrak_c1_30
    + 2 * b1 * mathfrak_c1_21
)

mathfrak_c2_12 = (
    -b1**2 * mathfrak_c2_30
    + 2 * b1 * mathfrak_c2_21
    + b2 * mathfrak_c1_20
)

mathfrak_c3_12 = (
    -b1**2 * mathfrak_c3_30
    + 2 * b1 * mathfrak_c3_21
    + b2 * mathfrak_c2_20
    + 2 * k**2 * b1 * (1 - chi) / b1**2 * b2
)

c3_12 = (
    -b1**2 * c3_30
    + 2 * b1 * c3_21
    + b2 * c3_20
    + k**2 * (
        2 * sp.diff((1-chi) * b2 / b1**2, y) * psi_y
        + sp.diff((1-chi) * b2 / b1**2, y, 2) * psi
        - (1-chi) / b1**2 * omega_yy
        )
    + k**2 * (
        -2 * sp.diff((1-chi) / b1**2, y) * omega_y
        + (
            k**2 * (1-chi) / b1**2
            - sp.diff((1-chi) / b1**2, y, 2)
            ) * omega
        )
)

c4_12 = (
    -2 * b2**2 * r30
    + 4 * b2 * sp.diff(r21, y)
    + 2 * b3 * r21
    + 2 * b3 * sp.diff(r20, y)
    + b4 * r20
    + chi * (
        (b2 * D2psi + 2 * b3 * Dpsi + b4 * psi - omega_yy) / denominator**2
        + (2 * b3 * sp.diff(Dpsi, lam) + b4 * psi_lam) / denominator
    )
    + 2 * chi_y * (
        (b2 * sp.diff(psi_lam, y) + b3 * psi_lam) / denominator
        + b2 * rayleigh_numerator / (b1 * denominator**2)
    )
    + chi_yy * b2 * psi_lam / denominator
)

R12_appendix = sp.expand(
    c1_12 * log_denominator / denominator
    + (
        c2_12
        + mathfrak_c1_12 * D2psi_y
        + mathfrak_c2_12 * Dpsi_y
        + mathfrak_c3_12 * psi_y
    ) / denominator
    + c3_12 * log_denominator
    + c4_12
)

# Apply reductions on R12_appendix.
R12_appendix_reduced = apply_raw_rayleigh_reductions(R12_appendix)

In [35]:
# Comparison 1: ask SymPy whether the reduced expressions are equal.
R12_matches_appendix = R12_direct_reduced.equals(R12_appendix_reduced)

print('For (a,b) = (1,2), SymPy .equals() returns:', R12_matches_appendix)

For (a,b) = (1,2), SymPy .equals() returns: True


In [36]:
# Comparison 2: compute the exact residual.
R12_difference = sp.cancel(R12_direct_reduced - R12_appendix_reduced)

print('For (a,b) = (1,2), the reduced direct-minus-appendix residual is:', R12_difference)

For (a,b) = (1,2), the reduced direct-minus-appendix residual is: 0


## Summary

In [37]:
# Summary:
remainder_checks = [
    ((1, 0), R10_direct_reduced, R10_appendix_reduced, R10_difference),
    ((2, 0), R20_direct_reduced, R20_appendix_reduced, R20_difference),
    ((1, 1), R11_direct_reduced, R11_appendix_reduced, R11_difference),
    ((3, 0), R30_direct_reduced, R30_appendix_reduced, R30_difference),
    ((2, 1), R21_direct_reduced, R21_appendix_reduced, R21_difference),
    ((1, 2), R12_direct_reduced, R12_appendix_reduced, R12_difference),
]
rewrite_checks = [
    ((1, 1), S11, S11_rewrite, S11_rewrite_difference),
    ((2, 1), S21, S21_rewrite, S21_rewrite_difference),
    ((1, 2), S12, S12_rewrite, S12_rewrite_difference),
]

for pair, direct, appendix, residual in remainder_checks:
    assert residual == 0, f'The remainder for {pair} has not been verified.'
    for expression in (direct, appendix):
        assert not expression.atoms(sp.Float), f'Floating-point coefficient found for {pair}.'
        assert not any(
            derivative.expr == psi and derivative.variables.count(y) >= 2
            for derivative in expression.atoms(sp.Derivative)
        ), f'An unreduced higher spatial derivative remains for {pair}.'
    print(f'Remainder {pair}: verified; exact residual = {residual}.')

for pair, original_singularity, rewritten_singularity, residual in rewrite_checks:
    assert residual == 0, f'The singular-term rewrite for {pair} has not been verified.'
    assert not original_singularity.atoms(sp.Float)
    assert not rewritten_singularity.atoms(sp.Float)
    print(f'Singular-term rewrite {pair}: verified; exact residual = {residual}.')

print('\nAll 6 remainder identities and all 3 singular-term rewrites are verified.')

Remainder (1, 0): verified; exact residual = 0.
Remainder (2, 0): verified; exact residual = 0.
Remainder (1, 1): verified; exact residual = 0.
Remainder (3, 0): verified; exact residual = 0.
Remainder (2, 1): verified; exact residual = 0.
Remainder (1, 2): verified; exact residual = 0.
Singular-term rewrite (1, 1): verified; exact residual = 0.
Singular-term rewrite (2, 1): verified; exact residual = 0.
Singular-term rewrite (1, 2): verified; exact residual = 0.

All 6 remainder identities and all 3 singular-term rewrites are verified.
